<a href="https://colab.research.google.com/github/ankitsingh435517/lichess-gpt/blob/main/lichess_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get install zstd
!pip install chess

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (501 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directo

In [ ]:
import subprocess
import io
import chess.pgn

N_GAMES = 10_000
OUTPUT_FILE = "games.txt"

# Download August 2026 Lichess database and stream-decompress it.
wget = subprocess.Popen(
    [
        "wget",
        "-qO-",
        "https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst",
    ],
    stdout=subprocess.PIPE,
)

zstd = subprocess.Popen(
    ["zstd", "-d"],
    stdin=wget.stdout,
    stdout=subprocess.PIPE,
)

# Allow wget to receive SIGPIPE if zstd exits early.
wget.stdout.close()

# Convert bytes → text so python-chess can parse PGN.
pgn_stream = io.TextIOWrapper(
    zstd.stdout,
    encoding="utf-8"
)

games_written = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    while games_written < N_GAMES:

        game = chess.pgn.read_game(pgn_stream)

        # End of database
        if game is None:
            break

        # Extract UCI moves
        uci_moves = [
            move.uci()
            for move in game.mainline_moves()
        ]

        # Ignore games with no moves
        if not uci_moves:
            continue

        # One complete game per line
        out.write(" ".join(uci_moves) + "\n")

        games_written += 1

        if games_written % 1000 == 0:
            print(f"Extracted {games_written}/{N_GAMES} games")

pgn_stream.close()

print(f"\nFinished.")
print(f"Games extracted: {games_written}")
print(f"Saved to: {OUTPUT_FILE}")

Extracted 1000/10000 games
Extracted 2000/10000 games
Extracted 3000/10000 games
Extracted 4000/10000 games
Extracted 5000/10000 games
Extracted 6000/10000 games
Extracted 7000/10000 games
Extracted 8000/10000 games
Extracted 9000/10000 games
Extracted 10000/10000 games

Finished.
Games extracted: 10000
Saved to: games.txt


In [ ]:
games = []
unique_uci_moves = set()

with open("games.txt", "r", encoding="utf-8") as f:
    for line in f:
        moves = line.strip().split()

        if moves:
            games.append(moves)
            unique_uci_moves.update(moves)

print("Games:", len(games))
print("Unique UCI moves:", len(unique_uci_moves))

Games: 10000
Unique UCI moves: 1858


In [ ]:
len(games)

10000

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
# full move uci tokenizer
vocab = ['.'] + sorted(set(unique_uci_moves))
vocab_size = len(sorted(vocab)) # all the unique moves are the vocabulary
move_to_id = {move:i for i, move in enumerate(vocab)}
id_to_move = {i:move for i, move in enumerate(vocab)}
encode = lambda moves: [move_to_id[move] for move in moves]
decode = lambda ids: ["".join(id_to_move[id] for id in ids)]

print(encode(["e2e4"]))
print(decode(encode(["e2e4"])))

[973]
['e2e4']


In [ ]:
import random
random.seed(42)
random.shuffle(games)

n1 = int(0.8 * len(games))
n2 = int(0.9 * len(games))

train_games = games[:n1]
val_games = games[n1:n2]
test_games = games[n2:]

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_batch_cpu(games, batch_size, context_size, move_to_id):
    X = torch.zeros(
        (batch_size, context_size),
        dtype=torch.long
    )

    Y = torch.zeros(
        batch_size,
        dtype=torch.long
    )

    for i in range(batch_size):
        game = games[torch.randint(len(games), (1,)).item()]
        pos = torch.randint(len(game), (1,)).item()
        context = [0] * context_size

        for move in game[:pos]:
            ix = move_to_id[move]
            context = context[1:] + [ix]

        X[i] = torch.tensor(context)
        Y[i] = move_to_id[game[pos]]

    return X, Y

In [ ]:
eval_iters = 200
batch_size=64
n_blocks = 6
n_embd = 96
n_head = 4
h_dim = 4 * n_embd
context_size = 128

eval_batches = {
    "train": [
        get_batch_cpu(train_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
    "val": [
        get_batch_cpu(val_games, batch_size, context_size, move_to_id)
        for _ in range(eval_iters)
    ],
}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
g = torch.Generator().manual_seed(1337)

class MLP(nn.Module):
  def __init__(self, n_embd, h_dim):
    super().__init__()
    self.hidden_layer = nn.Linear(n_embd, h_dim)
    self.relu = nn.ReLU()
    self.output_layer = nn.Linear(h_dim, n_embd)

  def forward(self, x):
    x = self.relu(self.hidden_layer(x))
    out = self.output_layer(x)
    return out

In [ ]:
import math

class Head(nn.Module):
  def __init__(self, n_embd, head_size, context_size):
    super().__init__()
    self.head_size = head_size
    self.Q = nn.Linear(n_embd, head_size)
    self.K = nn.Linear(n_embd, head_size)
    self.V = nn.Linear(n_embd, head_size)
    self.register_buffer(
        "tril",
        torch.tril(torch.ones(context_size, context_size)) # to match (B, T, T) each example will have T * T attention matrix where each row is a query and each column of that row is the key it needs to attend
    )

  def forward(self, x):
    q = self.Q(x)
    k = self.K(x)
    v = self.V(x)

    s = q @ k.transpose(-2, -1)
    s = s / math.sqrt(self.head_size)

    # causal mask
    mask = self.tril[:x.size(1), :x.size(1)]
    s = s.masked_fill(mask == 0, float("-inf"))

    w = F.softmax(s, dim=-1)

    o = w @ v

    return o

In [ ]:
class MHA(nn.Module):
  def __init__(self, n_embd, head_size, n_head, context_size):
    super().__init__()
    self.heads = nn.ModuleList([
        Head(n_embd, head_size, context_size) for _ in range(n_head)
    ])
    self.out_layer = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    outputs = [head(x) for head in self.heads]
    out = torch.cat(outputs, dim=-1)
    out = self.out_layer(out)
    return out

In [ ]:
class Block(nn.Module):
  def __init__(self, n_embd, n_head, h_dim, context_size):
    super().__init__()
    head_size = n_embd // n_head
    self.mha = MHA(n_embd, head_size, n_head, context_size)
    self.mlp = MLP(n_embd, h_dim)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x1 = x + self.mha(self.ln1(x))
    x2 = x1 + self.mlp(self.ln2(x1))
    return x2

In [ ]:
class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.blocks = nn.ModuleList([
        Block(n_embd, n_head, h_dim, context_size) for _ in range(n_blocks)
    ])
    self.embd_table = nn.Embedding(vocab_size, n_embd)
    self.pos_embd_table = nn.Embedding(context_size, n_embd)
    self.final_ln = nn.LayerNorm(n_embd)
    self.linear = nn.Linear(n_embd, vocab_size)

  def forward(self, x, y=None):
    tok_emb = self.embd_table(x)

    T = tok_emb.shape[1]
    pos_ids = torch.arange(T, device=x.device)
    pos_emb = self.pos_embd_table(pos_ids)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    x = self.final_ln(x)
    logits = self.linear(x)
    logits = logits[:, -1, :]
    if y is None:
      return logits
    loss = F.cross_entropy(logits, y)
    return logits, loss

In [ ]:
# model
model = GPT()

In [ ]:
model = model.to(device)

print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [ ]:
# optimizer
lr = 3e-4
wd = 0.01

total_params = sum(p.numel() for p in model.parameters())
print(total_params)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

1042307


In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

print("Device:", next(model.parameters()).device)

num_params = sum(p.numel() for p in model.parameters())
model_memory_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1024**2

print(f"Model parameters: {num_params / 1e6:.2f}M")
print(f"Model parameter memory: {model_memory_mb:.1f} MB")

print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
Device: cuda:0
Model parameters: 1.04M
Model parameter memory: 4.0 MB
Allocated: 0.23 GB
Reserved:  1.03 GB


In [ ]:
# train
import gc
import torch

# CUDA OOM Related ------------------------------
# Safely delete any older model and optimizer references to release the GPU memory
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer

# 1. Clean up lingering variables and clear GPU cache to recover the 14+ GB of wasted VRAM
gc.collect()
torch.cuda.empty_cache()

# Now, recreate the model and optimizer fresh in clean memory
from __main__ import GPT, device  # Ensure imports are present
model = GPT().to(device)

# Training code ----------------------------------
lr = 3e-4
wd = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

best_dev_loss = float("inf")
max_iters = 10000

# Safely adjust batch size if needed to avoid future OOMs
batch_size = 128  # Fits comfortably now within the cleared T4 memory

for step in range(max_iters):

    # -------------------------
    # Get a random training batch
    # -------------------------
    x, y = get_batch_cpu(
        train_games,
        batch_size,
        context_size,
        move_to_id
    )

    x = x.to(device)
    y = y.to(device)

    # -------------------------
    # Forward pass
    # -------------------------
    logits, loss = model(x, y)
    if step % 1000 == 0:
      print(loss.item())

    # -------------------------
    # Backward pass
    # -------------------------
    # set_to_none=True modestly reduces memory footprint during backward step
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

7.661985874176025
6.127402305603027
6.031181812286377
5.811739921569824
5.940450668334961
5.873373985290527
5.605299949645996
5.8729352951049805
6.003422737121582
5.837257385253906


In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Peak:      {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

Allocated: 0.23 GB
Reserved:  1.03 GB
Peak:      0.99 GB


In [ ]:
# evaluate train and val los
@torch.no_grad()
def estimate_loss():
    model.eval()

    out = {}

    for split in ["train", "val"]:
        total_loss = 0.0

        for X, Y in eval_batches[split]:
            X = X.to(device)
            Y = Y.to(device)

            _, loss = model(X, Y)
            total_loss += loss.item()

        out[split] = total_loss / len(eval_batches[split])

    model.train()
    return out

In [ ]:
estimate_loss()

{'train': 5.737158281803131, 'val': 5.740934598445892}

In [ ]:
def generate(max_new_tokens=50):
  context = [0] * context_size
  out = []
  for _ in range(max_new_tokens):
    # Create the tensor on the exact same device as the model (GPU)
    X = torch.tensor([context], device=device)
    logits = model(X)
    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    tok_id = next_id.item()
    out.append(tok_id)
    context = context[1:] + [tok_id]

    if tok_id == 0:
      break

  return out

generated_games = [decode(generate())[0] for _ in range(50)]
print(generated_games)

['e2e4d7d5e4d5a7a6f1c4e5d4b3a2e7e6g1f3d4e3b1d2e4f3b1c3d7d6f1b5c7c5d2e4d8h4h5e2f3e5b8c6h4g3g8e7h8h7c4a2b7b5e4e5h1g2c3c4c6e5h2h3c8g4a7b7f6f2d6c8h7h6c3d5e4f3h8c3g8h8d1e2f6a6c4b3d7d8e4g5e7f6e5e6c4d4b7b6c6a4', 'e2e4e7e5c2c4e8g8d3d4c5a7c1e3g8f6b3f3d8d5e4e5b8c6c2c3g4f3d2b3a6a5d2e4e8d8f3e5e8g8h2h3h5h3f8d8d7d2c1e3d8d6h1g1d8b6c3f3g7h6c2c3c8g4f3d4g8f6a8c7f3d2f8e7g8h8d2d6f6d5f4e4g1h1d5f7g7c3f3g5c3h8c2f5a8e8h4f6d3c3', 'e2e4e7e5c2c4b8c6g1f3d7d6d2d3g8f6d2d3g7g6d2d4e7e6f3f2c8g4g5f6b6c5b2c3e5e4e3g4d5f3d2d4b2b6g7c3f7e6a4a5e6f5f1e2b8d7b2a2c2d1c4d2c2g2e4e5d5f4h2h3d6e7b6c5d5f7g8h8b6c6c4a2d6e5g4h4a3e3a8d8e6e5a2a3b4d2e7c7f6f7', 'd2d4c5f2g1f3f8g7f1h3g8f6a2a3e8g8e4d5g8e7g1f3g5g4c1g5f8e7h2h3e8c8f4c1e6d7g2g3a6c8a2a3e8g8e3f2f7f6c3b5a5b6f6b6e1d2a8h8a1a8c7b6e6d8g7g6d2f3c6d5c2c3f1f8e5d4b2b3g5g6f1d3e5f6a6a3c8d8d1d3c6d5h2h3f6d7h2h1qf3e4', 'e2e4e7e5g1f3d7d6a1c1d7d6b1c3h7h6f1e2d7d6c2c4d7d6g1f3d7d6c1e3c8b7c7g3h8d8f1e1d6e5d3f5f8b4c1d2b1b2h8e8f1e1f6e4e3c5d6e5b2b4f5g4c1b1g4h5h8a8g6f5d5c6b4c5h6h5a2a4b4c3e8e1g4h4d7e5f2e3c2b4d

In [ ]:
import chess

positions = []

for moves in generated_games:
    moves = [moves[i:i+4] for i in range(0, len(moves), 4)]
    board = chess.Board()

    for i, move_uci in enumerate(moves):
        move = chess.Move.from_uci(move_uci)

        if move in board.legal_moves:
            board.push(move)
        else:
            positions.append(i + 1)
            break
    else:
        # Entire generated game was legal
        positions.append(len(moves) + 1)

In [ ]:
import numpy as np

positions = np.array(positions)

print("Mean first illegal position:", positions.mean())
print("Median:", np.median(positions))
print("Survive 10 moves:", np.mean(positions > 10))
print("Survive 20 moves:", np.mean(positions > 20))
print("Survive 50 moves:", np.mean(positions > 50))

Mean first illegal position: 3.66
Median: 3.0
Survive 10 moves: 0.0
Survive 20 moves: 0.0
Survive 50 moves: 0.0


In [ ]:
# -------
# Start the experiment on mech interp.
# -------
# Write up